## Imports

In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
%load_ext autoreload
%autoreload 2

## Load dataset

In [3]:
data_path = Path("../exploration/data/coffee_sales_prepared.parquet")

df = pd.read_parquet(data_path)
df.head()

,date,datetime,cash_type,card,money,coffee_name,hour,day_of_week,day_of_week_num,month,year,is_weekend,season
0,2024-03-01,2024-03-01 10:15:50.520,card,anon-0000-0000-0001,38.7,latte,10,Friday,4,3,2024,False,spring
1,2024-03-01,2024-03-01 12:19:22.539,card,anon-0000-0000-0002,38.7,hot chocolate,12,Friday,4,3,2024,False,spring
2,2024-03-01,2024-03-01 12:20:18.089,card,anon-0000-0000-0002,38.7,hot chocolate,12,Friday,4,3,2024,False,spring
3,2024-03-01,2024-03-01 13:46:33.006,card,anon-0000-0000-0003,28.9,americano,13,Friday,4,3,2024,False,spring
4,2024-03-01,2024-03-01 13:48:14.626,card,anon-0000-0000-0004,38.7,latte,13,Friday,4,3,2024,False,spring


## Daily series

In [4]:
daily_sales = (
    df.groupby("date")
    .size()
    .rename("sales_quantity")
    .to_frame()
    .sort_index()
)

daily_sales.index = pd.to_datetime(daily_sales.index)

daily_sales.head()

,sales_quantity
date,
2024-03-01,11
2024-03-02,7
2024-03-03,10
2024-03-04,4
2024-03-05,9


In [7]:
# confirmando se existe uma linha para cada dia
complete_dates = pd.date_range(
    start=daily_sales.index.min(),
    end=daily_sales.index.max(),
    freq="D"
)

missing_dates = complete_dates.difference(daily_sales.index)

print(f"Período: {daily_sales.index.min().date()} a "
      f"{daily_sales.index.max().date()}")

print(f"Quantidade de dias observados: {len(daily_sales)}")
print(f"Quantidade de dias ausentes: {len(missing_dates)}")

display(missing_dates)

Período: 2024-03-01 a 2025-03-23
Quantidade de dias observados: 381
Quantidade de dias ausentes: 7


DatetimeIndex(['2024-05-01', '2024-05-04', '2024-05-05', '2024-11-27',
               '2025-01-01', '2025-01-19', '2025-01-23'],
              dtype='datetime64[ns]', freq=None)

In [ ]:
daily_sales = daily_sales.asfreq("D")

daily_sales["was_missing"] = (
    daily_sales["sales_quantity"].isna()
)

daily_sales["sales_quantity"] = (
    daily_sales["sales_quantity"]
    .fillna(0)
    .astype(int)
)

In [7]:
daily_sales[daily_sales["was_missing"]]
# não entrará no modelo, apenas pra auditoria. pode indicar possível problema na máquina de vendas, por exemplo.

,sales_quantity,was_missing
date,,
2024-05-01,0,True
2024-05-04,0,True
2024-05-05,0,True
2024-11-27,0,True
2025-01-01,0,True
2025-01-19,0,True
2025-01-23,0,True


## Check incomplete registries

In [9]:
display(daily_sales.tail(10))

,sales_quantity
date,
2025-03-14,23
2025-03-15,7
2025-03-16,5
2025-03-17,13
2025-03-18,17
2025-03-19,21
2025-03-20,20
2025-03-21,20
2025-03-22,12


In [9]:
last_datetime = df["datetime"].max()

print(f"Última transação: {last_datetime}")
print(f"Última data: {last_datetime.date()}")
print(f"Horário da última transação: {last_datetime.time()}")

Última transação: 2025-03-23 18:11:38.635000
Última data: 2025-03-23
Horário da última transação: 18:11:38.635000


In [10]:
daily_summary = (
    df.groupby("date")
    .agg(
        sales_quantity=("coffee_name", "size"),
        first_transaction=("datetime", "min"),
        last_transaction=("datetime", "max")
    )
    .sort_index()
)

display(daily_summary.tail(10))

,sales_quantity,first_transaction,last_transaction
date,,,
2025-03-14,23,2025-03-14 07:55:05.952,2025-03-14 21:40:43.586
2025-03-15,7,2025-03-15 09:53:22.276,2025-03-15 19:51:27.089
2025-03-16,5,2025-03-16 09:32:17.307,2025-03-16 14:22:42.633
2025-03-17,13,2025-03-17 06:54:40.633,2025-03-17 16:46:44.575
2025-03-18,17,2025-03-18 08:23:10.985,2025-03-18 21:20:40.031
2025-03-19,21,2025-03-19 07:03:07.073,2025-03-19 18:31:25.990
2025-03-20,20,2025-03-20 08:13:16.283,2025-03-20 21:50:15.613
2025-03-21,20,2025-03-21 07:06:11.979,2025-03-21 18:59:24.271
2025-03-22,12,2025-03-22 09:05:13.059,2025-03-22 19:16:52.727


## Train-test split

In [11]:
forecast_horizon = 30

train = daily_sales.iloc[:-forecast_horizon].copy()
test = daily_sales.iloc[-forecast_horizon:].copy()

print(f"Treino: {train.index.min().date()} a "
      f"{train.index.max().date()}")

print(f"Teste: {test.index.min().date()} a "
      f"{test.index.max().date()}")

print(f"Observações de treino: {len(train)}")
print(f"Observações de teste: {len(test)}")

Treino: 2024-03-01 a 2025-02-21
Teste: 2025-02-22 a 2025-03-23
Observações de treino: 358
Observações de teste: 30


## Baseline seazoning

hipótese considerada: as vendas dos próximos dias serão semelhantes às vendas dos mesmos dias da última semana observada.